[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grabuffo/brain-modeling-course/blob/main/notebooks/D2_C4_NB08_hybrid_transitions_avalanches.ipynb)

# Day 2 · Chapter 4 · Notebook 08 — Three roads to synchrony, and where the brain-like avalanches are

**Goal.** Reproduce the main results of Buendía, Villegas, Burioni & Muñoz (2021) with the active-rotator population
of NB04/NB06:

1. the **phase diagram**: quiet, oscillating and asynchronous regimes;
2. the **three types of transition** into collective oscillations (type I / SNIC, type II / Hopf, and a **hybrid**
   type with bistability), in the mean field and in the network (**Fig. 3** of the paper);
3. **neuronal avalanches**: only at the hybrid transition are they scale-free, as in the cortex.

**Why this matters.** Since Beggs & Plenz (2003), cortical activity has been found to come in *avalanches*, cascades
of activity whose sizes and durations follow power laws. This is a hallmark of systems poised at a **critical point**,
the edge of a phase transition, and a popular hypothesis is that the brain works near such a point (Muñoz 2018). But
"near a transition" is not enough: as we will see, *which* transition matters.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this downloads the course code; on your laptop it does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("brain-modeling-course"):
    !git clone -q https://github.com/grabuffo/brain-modeling-course.git
    %cd brain-modeling-course/notebooks
    !pip -q install numba ipywidgets
sys.path.insert(0, os.path.abspath(".."))   # so that `import src` works

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from src import plotting
from src.plotting import slider
plotting.setup()
from src import networks, meanfield
CACHE = '../data/cache'

## 1. The phase diagram

Same network as NB04 ($N$ active rotators, $\omega=1$, $J=1$, noise $\sigma$, tilt $a$). For every $(a,\sigma)$ we run
the exact mean field (Kuramoto–Daido hierarchy, NB06) and classify the long-time state:

* **oscillations** (collective, "synchronous"): $Z(t)$ goes around a limit cycle, the population fires in bursts;
* **quiet** (*collective excitability*): a stable state with almost no firing; units sit near their rest phase;
* **asynchronous**: a stable state with high firing; units fire independently.

The grid takes about 1 minute, so it is precomputed in `data/cache` (the code that made it is `dev/cache_nb08_phase.py`;
if the file is missing, a coarser grid is computed here).

In [ ]:
path = os.path.join(CACHE, "nb08_phase_diagram.npz")
if os.path.exists(path):
    pdg = dict(np.load(path))
else:                                         # coarser grid, ~20 s
    A = np.round(np.arange(0.95, 1.1201, 0.01), 3); S = np.round(np.arange(0.2, 0.8001, 0.04), 3)
    res = np.array([[meanfield.kd_state(a, s, Z0=0.05 + 0j) for a in A] for s in S])
    pdg = {"A": A, "S": S, "rate_incoherent": res[..., 0], "osc_incoherent": res[..., 1]}
A_grid, S_grid = pdg["A"], pdg["S"]
regime = np.where(pdg["osc_incoherent"] > 0.05, 2, np.where(pdg["rate_incoherent"] < 0.005, 0, 1))  # 0 quiet, 1 async, 2 osc

PATHS = {"type II (Hopf)": dict(a=[1.0, 1.0, 1.0], s=[0.55, 0.67, 0.70], c="C3"),
         "hybrid": dict(a=[1.07, 1.07, 1.07], s=[0.52, 0.496, 0.48], c="C0"),
         "type I (SNIC)": dict(a=[1.02, 1.024, 1.03], s=[0.3, 0.3, 0.3], c="C2")}

def phase_diagram(ax):
    from matplotlib.colors import ListedColormap
    ax.pcolormesh(A_grid, S_grid, regime, shading="auto", cmap=ListedColormap(["#d9d9d9", "#9ecae1", "#fdae6b"]), vmin=0, vmax=2)
    ax.text(1.09, 0.3, "quiet\n(collective\nexcitability)", ha="center", fontsize=9)
    ax.text(0.99, 0.75, "asynchronous", ha="center", fontsize=9)
    ax.text(0.975, 0.4, "oscillations", ha="center", fontsize=9)
    for name, p in PATHS.items():
        ax.plot(p["a"], p["s"], "o-", c=p["c"], ms=5, lw=2, label=name)
    ax.axvline(1.0, c="k", ls=":", lw=0.8)
    ax.set(xlabel="tilt a   (single unit: oscillator if a < 1, excitable if a > 1)", ylabel="noise σ")

fig, ax = plt.subplots(figsize=(7.5, 5))
phase_diagram(ax)
ax.legend(fontsize=8, loc="lower left"); ax.set_title("phase diagram of the active-rotator population (ω = J = 1)")
plt.tight_layout()

The three segments are the three paths of Fig. 3 of the paper. Each one enters the oscillating regime from a different
side:

* **type II (red)**: from the asynchronous regime, by *decreasing* noise;
* **type I (green)**: from the quiet regime, by *decreasing* the tilt $a$;
* **hybrid (blue)**: from the quiet regime, by *increasing* noise, near the corner where the three regimes meet.

## 2. Three types of transition (mean field)

We now walk along each path slowly, in both directions, letting the mean field settle at every step and starting from
the previous state (*continuation*). Plotted: the mean firing rate and the amplitude of the collective oscillation.

In [ ]:
def continue_kd(a, s, Zk, T=400.0):
    # run the exact mean field at (a, s) starting from the modes Zk of the previous step
    K, dtk = meanfield.kd_setup(s)
    Z0 = np.zeros(K + 1, complex); n_ = min(K + 1, len(Zk)); Z0[:n_] = Zk[:n_]
    t, Z = meanfield.kuramoto_daido(a=a, sigma=s, T=T, K=K, dt=dtk, Zk0=Z0, return_modes=True)
    m = t > T / 2
    r = np.clip(meanfield.rotator_rate(Z[m], a=a, sigma=s), 0, None)
    z1 = Z[m, 1]
    osc = np.sqrt(np.mean(np.abs(z1 - z1.mean())**2))
    x = r - r.mean(); up = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]
    period = np.mean(np.diff(t[m][up])) if (len(up) > 2 and osc > 0.05) else np.nan
    full = np.zeros(61, complex); full[:K + 1] = Z[-1]
    return full, r.mean(), osc, period

def sweep(values, make_as, Zstart, T=400.0):
    Zk, out = Zstart, []
    for x in values:
        Zk, r, o, p = continue_kd(*make_as(x), Zk, T)
        out.append((r, o, p))
    return np.array(out), Zk

quiet0 = np.r_[1, (0.95 * np.exp(-0.5j * np.pi)) ** np.arange(1, 61)]    # start near the quiet state
incoh0 = np.r_[1, 0.05 ** np.arange(1, 61)]                               # start from incoherence

s2 = np.round(np.arange(0.55, 0.751, 0.01), 3)
s_h = np.round(np.arange(0.47, 0.5201, 0.0025), 4)
a1 = np.round(np.arange(1.0, 1.0301, 0.002), 3)
res = {}
res["II up"], Z = sweep(s2, lambda s: (1.0, s), incoh0); res["II down"], _ = sweep(s2[::-1], lambda s: (1.0, s), Z)
res["H up"], Z = sweep(s_h, lambda s: (1.07, s), quiet0); res["H down"], _ = sweep(s_h[::-1], lambda s: (1.07, s), Z)
res["I up"], Z = sweep(a1, lambda a: (a, 0.3), incoh0, T=800.0); res["I down"], _ = sweep(a1[::-1], lambda a: (a, 0.3), Z, T=800.0)

fig, ax = plt.subplots(2, 3, figsize=(15, 6.5), sharex="col")
for col, (key, x, lab, c) in enumerate([("II", s2, "noise σ   (a = 1.0)", "C3"), ("H", s_h, "noise σ   (a = 1.07)", "C0"), ("I", a1, "tilt a   (σ = 0.3)", "C2")]):
    up, dn = res[f"{key} up"], res[f"{key} down"][::-1]
    ax[0, col].plot(x, up[:, 0], "o-", c=c, ms=4, label="increasing"); ax[0, col].plot(x, dn[:, 0], "s--", c="k", mfc="none", ms=5, label="decreasing")
    ax[1, col].plot(x, up[:, 1], "o-", c=c, ms=4); ax[1, col].plot(x, dn[:, 1], "s--", c="k", mfc="none", ms=5)
    ax[0, col].set(ylabel="firing rate" if col == 0 else "", ylim=(0, None)); ax[1, col].set(xlabel=lab, ylabel="oscillation amplitude" if col == 0 else "")
ax[0, 0].set_title("type II (Hopf)"); ax[0, 1].set_title("hybrid"); ax[0, 2].set_title("type I (SNIC)")
ax[0, 1].legend(fontsize=8)
axp = ax[1, 2].twinx(); axp.plot(a1, res["I up"][:, 2], "^", c="C1", ms=5); axp.set_ylabel("period of the oscillation", color="C1")
plt.tight_layout()
bist = s_h[np.abs(res["H up"][:, 0] - res["H down"][::-1][:, 0]) > 0.003]
print("hybrid path: bistable for sigma in", bist)

* **Type II (Hopf).** Decreasing the noise from the asynchronous side, the oscillation is born with **zero amplitude**
  and grows continuously, while the rate changes smoothly: a supercritical Hopf bifurcation of the population (NB07).
* **Type I (SNIC).** Decreasing the tilt from the quiet side, the oscillation appears with a **finite amplitude** but
  an **infinitely long period** (orange triangles: the period grows as $a$ approaches the transition): the population
  version of the single QIF/rotator neuron of NB02.
* **Hybrid.** Increasing the noise from the quiet side, the firing rate **jumps**, and in a narrow window the result
  depends on the direction of the sweep: the population is **bistable** (quiet and active states coexist), like the
  subcritical Hopf of NB07. It is "hybrid" because it combines a jump (discontinuous, like a first-order transition) with
  oscillations that are born small (incipient oscillations), and, as we will see, critical-like avalanches.

## 3. The same three transitions in the network (Fig. 3)

Raster plots of 300 of $N=5000$ units, before, at and after each transition. The long simulations are precomputed
(`dev/cache_nb08.py`, ~10 min); if the cache is missing, shorter runs are done here (~1.5 min).

In [ ]:
CASES = {"hopf_sync": (1.0, 0.55, 300.0), "hopf_crit": (1.0, 0.67, 5000.0), "hopf_async": (1.0, 0.70, 300.0),
         "hybrid_quiet": (1.07, 0.48, 2000.0), "hybrid_crit": (1.07, 0.496, 5000.0), "hybrid_high": (1.07, 0.52, 2000.0),
         "snic_sync": (1.02, 0.3, 2000.0), "snic_crit": (1.024, 0.3, 5000.0), "snic_quiet": (1.03, 0.3, 2000.0)}
DT, N_NET = 0.01, 5000
path = os.path.join(CACHE, "nb08_network.npz")
if os.path.exists(path):
    net = dict(np.load(path))
else:
    net = {}
    for name, (a, s, T) in CASES.items():
        *_, counts, st, si = networks.active_rotator_network(N=N_NET, a=a, sigma=s, T=min(T, 500.0), dt=DT, rec_dt=10.0,
                                                             n_rec=1, seed=1, return_spikes=True, n_spk=300)
        net[f"counts_{name}"], net[f"spk_t_{name}"], net[f"spk_i_{name}"] = counts, st, si

fig, ax = plt.subplots(3, 3, figsize=(15, 9))
rows = [("hopf", ["sync", "crit", "async"], 100.0), ("hybrid", ["high", "crit", "quiet"], 2000.0), ("snic", ["sync", "crit", "quiet"], 2000.0)]
for r, (fam, keys, tmax) in enumerate(rows):
    for c, k in enumerate(keys):
        name = f"{fam}_{k}"; a, s, T = CASES[name]
        st, si = net[f"spk_t_{name}"], net[f"spk_i_{name}"]
        tmax_ = min(tmax, len(net[f"counts_{name}"]) * DT)
        keep = st < tmax_
        ax[r, c].plot(st[keep], si[keep], ".k", ms=0.5 if tmax > 500 else 1.0)
        ax[r, c].set(xlim=(0, tmax_), title=f"a = {a}, σ = {s}", ylabel="unit #" if c == 0 else "")
        if r == 2: ax[r, c].set_xlabel("time")
for r, lab in enumerate(["type II (Hopf)", "hybrid", "type I (SNIC)"]):
    ax[r, 0].annotate(lab, xy=(-0.28, 0.5), xycoords="axes fraction", rotation=90, va="center", fontsize=12, color=["C3", "C0", "C2"][r])
plt.tight_layout()

Columns as in the paper: on the active side of the transition, at the transition, on the other side. Note the
different time axes (100 vs 2000 time units).

* **Type II**: regular, fast population bursts that weaken into irregular activity as noise increases.
* **Type I**: long silences broken by **huge, all-or-none population bursts**; near the transition the silences become
  very long (the diverging period).
* **Hybrid**: at the transition the network alternates irregularly between quiet periods and episodes of collective
  activity of **all sizes**: some bursts die out after a few spikes, some recruit large parts of the network.

## 4. Neuronal avalanches

**Protocol** (Beggs & Plenz 2003, as adapted by Buendía et al.):
1. a unit "spikes" when its phase crosses $\pi/2$;
2. time is divided in bins of width $\Delta t$ = the mean interval between consecutive spikes of the whole population;
3. an **avalanche** is a run of consecutive non-empty bins between two empty bins; its **size** $S$ is the number of
   spikes, its **duration** $D$ the number of bins.

At a critical point, $P(S)\sim S^{-\tau}$ and $P(D)\sim D^{-\alpha}$, with no characteristic scale, and the mean size of
avalanches of duration $D$ grows as $\langle S\rangle(D)\sim D^{\gamma}$ with $\gamma = (\alpha-1)/(\tau-1)$ (*crackling
noise* relation, a stronger test than a single power law).

In [ ]:
def avalanches(counts, dt, bin_size):
    k = max(1, int(round(bin_size / dt)))
    c = counts[: len(counts) // k * k].reshape(-1, k).sum(axis=1)
    edges = np.diff(np.r_[0, (c > 0).astype(int), 0])
    start, end = np.where(edges == 1)[0], np.where(edges == -1)[0]
    cs = np.r_[0, np.cumsum(c)]
    return cs[end] - cs[start], end - start, c          # sizes, durations (in bins), binned activity

def log_hist(x, n_bins=25):
    b = np.unique(np.round(np.logspace(0, np.log10(x.max() + 1), n_bins)))
    h, _ = np.histogram(x, b)
    m = np.sqrt(b[:-1] * b[1:])
    return m[h > 0], (h / np.diff(b) / len(x))[h > 0]

def tau_mle(S, s_min=10):
    # maximum-likelihood power-law exponent for S >= s_min (discrete approximation, Clauset et al. 2009)
    x = S[S >= s_min]
    return 1 + len(x) / np.sum(np.log(x / (s_min - 0.5)))

aval = {}
for name in ["hopf_crit", "hybrid_crit", "snic_crit", "hybrid_quiet"]:
    counts = net[f"counts_{name}"].astype(int)[int(200 / DT):]           # skip the first 200 time units
    iei = len(counts) * DT / counts.sum()
    S, D, c = avalanches(counts, DT, iei)
    aval[name] = dict(S=S, D=D, iei=iei, c=c)

# what the avalanches look like: binned activity at the hybrid transition
c = aval["hybrid_crit"]["c"]; iei = aval["hybrid_crit"]["iei"]
tb = np.arange(len(c)) * iei
fig, ax = plt.subplots(figsize=(13, 2.6))
empty = np.convolve(c == 0, np.ones(200) / 200, "same")              # fraction of empty bins nearby
t_on = tb[np.argmin(np.abs(empty - 0.3))]                            # a stretch where avalanches are visible
sel = (tb > t_on - 3) & (tb < t_on + 3)
ax.fill_between(tb[sel], c[sel], step="mid", color="C0", lw=0)
ax.set(xlabel="time", ylabel="spikes per bin", title=f"hybrid transition: activity in bins of Δt = {iei:.3f} — avalanches are separated by empty bins")
plt.tight_layout()

In [ ]:
labels = {"hopf_crit": ("type II (Hopf)", "C3"), "hybrid_crit": ("hybrid", "C0"), "snic_crit": ("type I (SNIC)", "C2"),
          "hybrid_quiet": ("quiet (a = 1.07, σ = 0.48)", "0.5")}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for name, (lab, c) in labels.items():
    S, D = aval[name]["S"], aval[name]["D"]
    m, p = log_hist(S); ax[0].loglog(m, p, "o-", c=c, ms=4, lw=1, label=lab)
    m, p = log_hist(D); ax[1].loglog(m, p, "o-", c=c, ms=4, lw=1)
    if name != "hybrid_quiet":
        Du = np.unique(D); ax[2].loglog(Du, [S[D == u].mean() for u in Du], ".", c=c, ms=4)
x = np.logspace(0.3, 3.3)
ax[0].loglog(x, 0.5 * x**-2.1, "k--", lw=1, label=r"$S^{-2.1}$ (paper)")
ax[1].loglog(x[:30], 0.5 * x[:30]**-2.5, "k--", lw=1, label=r"$D^{-2.5}$ (paper)")
g = (2.5 - 1) / (2.1 - 1)
ax[2].loglog(x[:30], x[:30]**g, "k--", lw=1, label=fr"$D^{{{g:.2f}}}$ = $D^{{(\alpha-1)/(\tau-1)}}$")
ax[0].set(xlabel="avalanche size S", ylabel="P(S)", title="sizes"); ax[0].legend(fontsize=8)
ax[1].set(xlabel="avalanche duration D (bins)", ylabel="P(D)", title="durations"); ax[1].legend(fontsize=8)
ax[2].set(xlabel="duration D (bins)", ylabel="mean size <S>(D)", title="crackling-noise scaling"); ax[2].legend(fontsize=8)
plt.tight_layout()
for name, (lab, _) in labels.items():
    S = aval[name]["S"]
    print(f"{lab:28s}: {len(S):6d} avalanches, largest {S.max():5d} spikes, power-law exponent (S >= 10) {tau_mle(S):.2f}")

* **Hybrid**: sizes and durations are power laws over two to three decades, with exponents close to those of the paper
  ($\tau\approx2.1$, $\alpha\approx2.5$), and the mean size scales with duration as the crackling-noise relation predicts:
  three independent signatures of criticality. (The bump at the far end is the occasional system-wide burst; in the paper
  it scales with $N$.)
* **Type II (Hopf)**: a characteristic avalanche size (a hump, then a fast fall-off): the regular bursts of the raster.
* **Type I (SNIC)**: two kinds of events, tiny avalanches and **system-spanning** bursts of size $\approx N$, nothing in
  between.
* **Quiet**: only small avalanches.

A complementary signature: how irregular is each unit's firing? The coefficient of variation of the inter-spike
intervals is $CV=0$ for a clock, $1$ for a Poisson process, $>1$ for bursty, irregular firing (as in the cortex).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for name in ["hopf_crit", "hybrid_crit", "snic_crit"]:
    st, si = net[f"spk_t_{name}"], net[f"spk_i_{name}"]
    cvs = [np.std(np.diff(st[si == i])) / np.mean(np.diff(st[si == i])) for i in range(300) if (si == i).sum() > 10]
    ax.hist(cvs, bins=np.linspace(0, 2, 41), color=labels[name][1], alpha=0.6, label=f"{labels[name][0]}: median CV = {np.median(cvs):.2f}")
ax.axvline(1, c="k", ls=":"); ax.set(xlabel="CV of the inter-spike intervals (one value per unit)", ylabel="units")
ax.legend(fontsize=8); plt.tight_layout()

Only at the hybrid transition is single-unit firing irregular ($CV>1$), as observed in cortical neurons; at the Hopf
and SNIC transitions units fire more regularly than a Poisson process.

**Why the hybrid transition?** At the edge of the bistable region the population sits in the quiet state, but the active
state is very close. Noise starts small cascades; each cascade pushes the population toward the active state, where
firing recruits more firing. Most cascades die; some grow; a few take over the network before the population falls back.
Bistability plus noise plus a nearby oscillation gives cascades of **all sizes**. At a Hopf transition there is no nearby
"other state" to escape to, and at a SNIC every escape is complete (all-or-none).

> **Careful with power laws.** A straight line on a log-log plot is not proof of criticality: power laws can arise
> without a critical point (e.g. Touboul & Destexhe 2017). That is why we checked several signatures (two exponents, their
> scaling relation, and in the paper, finite-size scaling). The position of the transition also shifts with $N$
> (paper: $\sigma_c = 0.520, 0.505, 0.496$ for $N = 500, 1000, 5000$).

### Explore: find the avalanches

$N=1000$ units, $T=500$ (about 2 s per run). With $N=1000$ the hybrid transition sits near $\sigma\approx0.505$ at $a=1.07$.

* Start in the quiet state ($a=1.07$, $\sigma=0.48$) and increase $\sigma$ in small steps. Where do avalanches of all
  sizes appear? What happens beyond?
* Go to the Hopf path ($a=1.0$) and the SNIC path ($\sigma=0.3$): can you find power laws there?
* Careful: with short runs ($T=500$) almost any distribution looks straight for small sizes. Judge by the tail: how
  large is the largest avalanche, and does the curve follow the dashed line over two decades or bend down?

In [ ]:
@interact(a=slider(1.07, 0.95, 1.12, 0.005, "tilt a"), sigma=slider(0.505, 0.2, 0.8, 0.0025, "noise σ"))
def explore_avalanches(a, sigma):
    Nx, Tx = 1000, 500.0
    t, Z, ph, counts, st, si = networks.active_rotator_network(N=Nx, a=a, sigma=sigma, T=Tx, dt=DT, rec_dt=1.0, n_rec=1,
                                                               seed=2, return_spikes=True, n_spk=100)
    counts = counts[int(50 / DT):]
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8), gridspec_kw=dict(width_ratios=[2, 1, 1]))
    ax[0].plot(st, si, ".k", ms=1); ax[0].set(xlabel="time", ylabel="unit #", xlim=(0, Tx))
    if counts.sum() < 50:
        ax[0].set_title(f"almost silent ({counts.sum()} spikes)"); plt.tight_layout(); plt.show(); return
    iei = len(counts) * DT / counts.sum()
    S, D, _ = avalanches(counts, DT, iei)
    ax[0].set_title(f"rate {counts.sum() / Nx / (Tx - 50):.4f} per unit,  {len(S)} avalanches")
    m, p = log_hist(S); ax[1].loglog(m, p, "o-", ms=4)
    x = np.logspace(0, 3); ax[1].loglog(x, 0.5 * x**-2.1, "k--", lw=1, label=r"$S^{-2.1}$")
    ax[1].set(xlabel="size S", ylabel="P(S)", title=f"largest avalanche {S.max()}"); ax[1].legend(fontsize=8)
    phase_diagram(ax[2]); ax[2].plot(a, sigma, "k*", ms=14); ax[2].set(xlabel="a", ylabel="σ")
    plt.tight_layout(); plt.show()

## 5. Take-home

* The active-rotator population has three regimes (quiet, oscillating, asynchronous) and three different ways to enter
  synchrony: **type I** (SNIC: infinite period, all-or-none bursts), **type II** (Hopf: oscillations born small and
  regular) and **hybrid** (a jump with bistability and incipient oscillations).
* Only at the **hybrid** transition does the network show the signatures observed in cortex: **scale-free avalanches**
  that obey the crackling-noise relation, and **irregular single-unit firing** ($CV>1$).
* "Being near a transition" is not enough to be brain-like: the *type* of transition matters. The mean field tells us
  where each type lives; the network shows what it looks like.
* **Next (Chapter 5):** put such populations on a human connectome and ask which regime, and which distance from the
  transition, reproduces whole-brain data.

### Homework
*To do at home, after the class. In class we use the **Explore** sliders; these exercises take the same
questions further and usually need a few lines of your own code.*
1. **Finite size.** Run the network at the hybrid transition for $N=500$ ($\sigma=0.520$) and $N=1000$ ($\sigma=0.505$)
   (paper values) and compare $P(S)$. Where is the bump of system-wide avalanches?
2. **Bin size.** Recompute $P(S)$ at the hybrid transition with bins $\Delta t/2$ and $2\Delta t$. Does the exponent
   change? (It does, somewhat: this is a known issue when comparing exponents across experiments.)
3. **Away from the transition.** Compute $P(S)$ and the exponent at $a=1.07$ for $\sigma=0.49$, $0.50$, $0.51$ (use the
   cache-making code with shorter runs). How quickly does the power law disappear?
4. **Bistability in the network.** At $a=1.07$, $\sigma=0.494$, $N=5000$, run twice: once starting from all units at the
   rest phase ($\varphi_0=-\arcsin(1/a)$ plus small noise), once from random phases. Do the two runs look different for
   the first few hundred time units?
5. **Kuramoto.** Repeat the avalanche analysis for the noisy Kuramoto model ($a=0$) at its transition ($J=1$,
   $\sigma=1$, NB04). Are there scale-free avalanches?

### References
* Buendía V, Villegas P, Burioni R, Muñoz MA (2021). Hybrid-type synchronization transitions: where incipient oscillations, scale-free avalanches, and bistability live together. *Phys Rev Research* 3:023224.
* Beggs JM, Plenz D (2003). Neuronal avalanches in neocortical circuits. *J Neurosci* 23:11167–11177.
* Muñoz MA (2018). Colloquium: Criticality and dynamical scaling in living systems. *Rev Mod Phys* 90:031001.
* di Santo S, Villegas P, Burioni R, Muñoz MA (2018). Landau–Ginzburg theory of cortex dynamics: scale-free avalanches emerge at the edge of synchronization. *PNAS* 115:E1356–E1365.
* Friedman N et al. (2012). Universal critical dynamics in high resolution neuronal avalanche data. *Phys Rev Lett* 108:208102.
* Touboul J, Destexhe A (2017). Power-law statistics and universal scaling in the absence of criticality. *Phys Rev E* 95:012413.
* Clauset A, Shalizi CR, Newman MEJ (2009). Power-law distributions in empirical data. *SIAM Rev* 51:661–703.